In [1]:
# IMPORTS
import urllib3
import pandas as pd
import requests
from io import StringIO
from datetime import datetime
import os

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
# FUNÇÃO INPUT
def input_bronze(dataset: str) -> pd.DataFrame:
    url = f"https://dados.anvisa.gov.br/dados/VigiMed_{dataset}.csv"
    resp = requests.get(url, verify=False)
    data = StringIO(resp.text)
    df = pd.read_csv(
        data,
        sep=';',
        encoding='latin-1',
        dtype=str,
        quoting=3,
        on_bad_lines='error',
        low_memory=False
    )
    return df

# FUNÇÃO OUTPUT
def output_bronze(df: pd.DataFrame, dataset: str) -> str:
    layer = "01_bronze"
    folder = os.path.join("..", "data", layer, dataset)  # subpasta por dataset
    os.makedirs(folder, exist_ok=True)

    # remove todos os arquivos antigos da pasta
    for f in os.listdir(folder):
        file_path = os.path.join(folder, f)
        if os.path.isfile(file_path):
            os.remove(file_path)

    # adiciona coluna de data/hora atual no final
    df["ATUALIZADO"] = datetime.now().strftime("%Y-%m-%d")
    output_path = os.path.join(folder, f"{dataset}.parquet")
    df.to_parquet(output_path, index=False, engine="pyarrow")
    return output_path

In [3]:
# EXECUÇÃO
for dataset in ["Medicamentos", "Notificacoes", "Reacoes"]:
    df = input_bronze(dataset)
    path = output_bronze(df, dataset)